In [1]:
from pathlib import Path
from typing import Any
import shutil
import chromadb
import numpy as np
import pymupdf
import torch

from langchain_text_splitters import RecursiveCharacterTextSplitter
from ollama import chat
from sentence_transformers import SentenceTransformer

c:\Users\AIML\miniconda3\envs\aiml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CURRENT_DIRECTORY = Path.cwd()

print(CURRENT_DIRECTORY)

PROJECT_DIRECTORY = CURRENT_DIRECTORY.parent

print(PROJECT_DIRECTORY)


PDF_PATH = PROJECT_DIRECTORY / 'data' / 'arogya_shield_policy_handbook.pdf'

CHROMA_DB = PROJECT_DIRECTORY / 'chromadb'

print(CHROMA_DB)

c:\Users\AIML\rag_pipeline\python_notebook
c:\Users\AIML\rag_pipeline
c:\Users\AIML\rag_pipeline\chromadb


In [3]:
pages: list[dict[str, Any]] = []

In [4]:
print(type(pages))

<class 'list'>


In [5]:
with pymupdf.open(PDF_PATH) as document:
    for page_index, page in enumerate(document):
        page_text = page.get_text('text', sort = True).strip()

        if not page_text:
            continue

        pages.append(
            {
                'text': page_text,
                'metadata': {
                    'source': PDF_PATH.name,
                    'page_number': page_index + 1,
                    'document_title': (
                        'Arogya shield member policy handbook'
                    ),
                    'document_version': '0.1',
                },
            }
        )


len(pages)

8

In [6]:
pages[0]['text']

'Arogya Shield\n\n Residential Care Services Pvt. Ltd.\n\n\nMember Policy Handbook\n\n Version 3.2 — Effective 1 April 2026\n\n\n\n\n Arogya Shield Residential Care Services provides round-the-clock emergency medical\n response, paramedic support, and preventive health services to residents of registered\n gated communities in Hyderabad. This handbook describes membership plans,\n emergency response commitments, refund and cancellation rules, and grievance\n procedures applicable to all members.\n\n This is a fictional document created for educational demonstration purposes. Any\n resemblance to a real organisation is coincidental.\n\n\n\n\n\nArogya Shield Residential Care Services — Member Policy Handbook v3.2                           Page 1'

In [7]:
pages[1]['metadata']

{'source': 'arogya_shield_policy_handbook.pdf',
 'page_number': 2,
 'document_title': 'Arogya shield member policy handbook',
 'document_version': '0.1'}

In [8]:
pages[1]['text']

'1. Membership Plans and Pricing\n\n Arogya Shield offers three membership tiers. All prices are per household per month and\n include Goods and Services Tax.\n\n\n Plan              Monthly Fee   Included Services\n\n                                  Emergency response, ambulance dispatch, 1 free\n Shield Basic          ₹1,200        home nurse visit per month\n\n                                        Everything in Basic, plus quarterly full-body\n Shield Plus           ₹2,900            vitals screening and 24×7 tele-doctor line\n\n                                        Everything in Plus, plus a dedicated on-call\n Shield Premium       ₹4,800         paramedic and 4 physiotherapy sessions per month\n\n\n A one-time enrolment fee of ₹999 applies to all plans and is waived for communities with\n 200 or more registered households. Senior citizens aged 70 and above receive a 15\n percent discount on the Shield Premium plan.\n\n Physiotherapy sessions included with the Shield Premium p

## Part 2 - Chunking

In [9]:
example_text = (
    "The advanced life-support ambulance should arrive within 15 minutes. "
    "A documented service failure occurs when the response time exceeds "
    "twice the committed limit. Eligible members receive a refund of that "
    "month's membership fee."
)

print(example_text)

The advanced life-support ambulance should arrive within 15 minutes. A documented service failure occurs when the response time exceeds twice the committed limit. Eligible members receive a refund of that month's membership fee.


In [10]:
print(len(example_text))

228


## Naive fixed size chunking

In [11]:
chunk_size = 70

chunks = [example_text[start:start + chunk_size] for start in range(0, len (example_text), chunk_size)]

for i in range(len(chunks)):
    print(len(chunks[i]))

70
70
70
18


In [12]:
chunks[0]

'The advanced life-support ambulance should arrive within 15 minutes. A'

In [13]:
chunks[1]

' documented service failure occurs when the response time exceeds twic'

In [14]:
chunks[2]

'e the committed limit. Eligible members receive a refund of that month'

In [15]:
chunks[3]

"'s membership fee."

## Recursive text splitter

In [16]:
CHUNK_SIZE = 350
CHUNK_OVERLAP = 80

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = CHUNK_SIZE,
    chunk_overlap = CHUNK_OVERLAP,
    length_function = len,
    add_start_index = True,
    separators = [
        '\n\n',
        '\n',
        '.',
        ' ',
        '',
    ],
    
)

In [17]:
example_texts = """
Arogya Shield provides emergency medical support to registered members.

An advanced life-support ambulance should arrive within 15 minutes. If the response time exceeds twice the committed limit, the incident may qualify as a documented service failure.

Eligible members may receive a refund of that month's membership fee.
"""


chunks = text_splitter.split_text(example_text)

for index, chunk in enumerate(chunks, start = 1):
    print(index)
    print(len(chunk))
    print(chunk)


1
228
The advanced life-support ambulance should arrive within 15 minutes. A documented service failure occurs when the response time exceeds twice the committed limit. Eligible members receive a refund of that month's membership fee.


In [18]:
pages[0]['text']

'Arogya Shield\n\n Residential Care Services Pvt. Ltd.\n\n\nMember Policy Handbook\n\n Version 3.2 — Effective 1 April 2026\n\n\n\n\n Arogya Shield Residential Care Services provides round-the-clock emergency medical\n response, paramedic support, and preventive health services to residents of registered\n gated communities in Hyderabad. This handbook describes membership plans,\n emergency response commitments, refund and cancellation rules, and grievance\n procedures applicable to all members.\n\n This is a fictional document created for educational demonstration purposes. Any\n resemblance to a real organisation is coincidental.\n\n\n\n\n\nArogya Shield Residential Care Services — Member Policy Handbook v3.2                           Page 1'

In [19]:
all_chunks: list[dict[str, Any]] = []

for page in pages:
    page_documents = text_splitter.create_documents(
        texts = [page['text']],
        metadatas = [page['metadata']]
    )

    for chunk_index, document in enumerate(page_documents, start = 1):
        page_number = int(document.metadata['page_number'])

        chunk_id = (
            f"{Path(document.metadata['source']).stem}"
            f'-p{page_number: 02d}'
            f'-c{chunk_index:02d}'
        )

        metadata = dict(document.metadata)
        metadata['chunk_index_on_page'] = chunk_index
        metadata['chunk_id'] = chunk_id
        metadata['character_count'] = len(document.page_content)

        all_chunks.append(
            {
                'text': document.page_content.strip(),
                'metadata': metadata,
            }
        )


print(all_chunks)




[{'text': 'Arogya Shield\n\n Residential Care Services Pvt. Ltd.\n\n\nMember Policy Handbook\n\n Version 3.2 — Effective 1 April 2026', 'metadata': {'source': 'arogya_shield_policy_handbook.pdf', 'page_number': 1, 'document_title': 'Arogya shield member policy handbook', 'document_version': '0.1', 'start_index': 0, 'chunk_index_on_page': 1, 'chunk_id': 'arogya_shield_policy_handbook-p 1-c01', 'character_count': 115}}, {'text': 'Arogya Shield Residential Care Services provides round-the-clock emergency medical\n response, paramedic support, and preventive health services to residents of registered\n gated communities in Hyderabad. This handbook describes membership plans,\n emergency response commitments, refund and cancellation rules, and grievance', 'metadata': {'source': 'arogya_shield_policy_handbook.pdf', 'page_number': 1, 'document_title': 'Arogya shield member policy handbook', 'document_version': '0.1', 'start_index': 121, 'chunk_index_on_page': 2, 'chunk_id': 'arogya_shield_pol

In [20]:
print(len(pages))
print(len(all_chunks))

8
35


In [21]:
page_three_chunks = [
    chunk
    for chunk in all_chunks
    if chunk['metadata']['page_number'] == 3
]

for chunk in page_three_chunks:
    print('chunk id:', chunk['metadata']['chunk_id'])
    print('start index', chunk['metadata'].get('start_index'))
    print('-' * 100)
    print(chunk['text'])

chunk id: arogya_shield_policy_handbook-p 3-c01
start index 0
----------------------------------------------------------------------------------------------------
2. Emergency Response Commitments

 Our response commitments below apply to all membership tiers within registered
 community premises.

    • First responder at the resident's door within 8 minutes of an emergency call.

    • Advanced life support ambulance on site within 15 minutes.
chunk id: arogya_shield_policy_handbook-p 3-c02
start index 226
----------------------------------------------------------------------------------------------------
• Advanced life support ambulance on site within 15 minutes.

    • Handover to a partner hospital emergency department within 40 minutes for critical
    cases.

 Every registered tower is equipped with one automated external defibrillator (AED)
 placed in the ground-floor lobby, inspected on the first Monday of every month.
chunk id: arogya_shield_policy_handbook-p 3-c03
start ind

## Create embeddings

In [22]:
EMBEDDIN_MODEL_NAME = (
    'sentence-transformers/'
    'multi-qa-MiniLM-l6-cos-v1'
)

embedding_model = SentenceTransformer(
    EMBEDDIN_MODEL_NAME,
    device = 'cpu'
)

print('Embedding  model', EMBEDDIN_MODEL_NAME)
print('Embedding dimensions', embedding_model.get_embedding_dimension())
print('maximum sequence length', embedding_model.max_seq_length)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8042.69it/s]


Embedding  model sentence-transformers/multi-qa-MiniLM-l6-cos-v1
Embedding dimensions 384
maximum sequence length 512


In [23]:
chunk_texts = [
    chunk['text']
    for chunk in all_chunks

]

chunk_embeddings = embedding_model.encode_document(
    chunk_texts,
    batch_size= 32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)




Batches: 100%|██████████| 2/2 [00:00<00:00,  7.92it/s]


In [24]:
print('Embedding shape:', chunk_embeddings.shape)

Embedding shape: (35, 384)


In [25]:
print('Embedding type:', type(chunk_embeddings))

Embedding type: <class 'numpy.ndarray'>


## Manual semantic search

In [26]:
def manual_semantic_search(
        question: str,
        chunks: list[dict[str, Any]],
        document_embeddings: np.ndarray,
        model: SentenceTransformer,
        top_k: 5,
) -> list[dict[str, Any]]:

    question = question.strip()

    if not question:
        raise ValueError('Question cannot be empty')

    query_embedding = model.encode_query(
        question, 
        convert_to_numpy = True,
        normalize_embeddings= True,
    )

    scores = document_embeddings @ query_embedding


    top_k = min(top_k, len(all_chunks))

    top_indices = np.argsort(scores)[::-1][:top_k]

    results: list[dict[str, Any]] = []

    for rank, index in enumerate(
        top_indices, start = 1):
        chunk = all_chunks[int(index)]

        results.append(
            {
                'rank': rank,
                'score': float(scores[index]),
                'text': chunk['text'],
                'metadata': dict(chunk['metadata'])

            }
        )

    return results


    

In [27]:
def display_results(
        results: list[dict[str, Any]],
) -> None:
    for result in results:
        print('Rank', result['rank'])
        print('Similarity score', f"{result['score']:.4f}")
        print("page", result['metadata']['page_number'])
        print(result['text'])

In [28]:
physiptherapy_results = manual_semantic_search(
    question = (
        'Can i use my unused physiotherapy'
        'Sessions during the next month?'
    ),
    chunks = chunks,
    document_embeddings = chunk_embeddings,
    model = embedding_model,
    top_k = 5,
)

display_results(physiptherapy_results)

Rank 1
Similarity score 0.6476
page 2
Physiotherapy sessions included with the Shield Premium plan must be used within the
 applicable billing month. Unused sessions cannot be carried forward, transferred to
 another household, or converted into a cash benefit.

 1.1 Billing Cycle
Rank 2
Similarity score 0.3870
page 2
Everything in Basic, plus quarterly full-body
 Shield Plus           ₹2,900            vitals screening and 24×7 tele-doctor line

                                        Everything in Plus, plus a dedicated on-call
 Shield Premium       ₹4,800         paramedic and 4 physiotherapy sessions per month
Rank 3
Similarity score 0.3301
page 2
1.1 Billing Cycle

 Membership fees are billed on the 5th of every month. A grace period of 10 days applies,
 after which services are suspended until payment is received. Reactivation after
 suspension carries a fee of ₹250.





Arogya Shield Residential Care Services — Member Policy Handbook v3.2                           Page 2
Rank 4

## Store the vectors in a vector database
## Multi-query retrieval 
## Build the grounded prompt 
## Generate the answer using the local LLM Qwen 3.5

In [29]:
chroma_client = chromadb.PersistentClient(
    path = str(CHROMA_DB)
)

collection = chroma_client.get_or_create_collection(
    name = 'arogya_shield_policy',
    metadata = {'hnsw:space': 'cosine'}
)

print('Collection created:', collection.name)

Collection created: arogya_shield_policy


In [30]:
chroma_ids = [
    chunk['metadata']['chunk_id']
    for chunk in all_chunks
]

chroma_documents = [
    chunk['text'] for chunk in all_chunks
]

chroma_metadata = [
    {
        key: value
        for key, value in chunk['metadata'].items()
        if isinstance(value, (str, int, float, bool))
    }
    for chunk in all_chunks
]

collection.add(
    ids = chroma_ids,
    documents = chroma_documents,
    embeddings = chunk_embeddings.tolist(),
    metadatas = chroma_metadata,
)

print('Items stored in Chroma:', collection.count())

Items stored in Chroma: 35


In [31]:
assert collection.count() == len(all_chunks)


## Query chroma

In [32]:
question = 'Which number i should call at 11PM for non-critical medical guidance'

query_embedding = embedding_model.encode_query(
    question,
    convert_to_numpy = True,
    normalize_embeddings = True,
)

query_result = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=min(5, collection.count()),
    include = [
        'documents',
        'metadatas',
        'distances',
    ],
)

documents = query_result['documents'][0]
metadata = query_result['metadatas'][0]
print(documents)
print(metadata)
distances = query_result['distances'][0]
print(distances)

['7. Contact Directory\n\n\n Purpose                     Contact                       Hours\n\n Medical emergency              1800-4032-108                  24 × 7\n\n Night helpline (non-critical)       1800-4032-777                  8 PM – 8 AM\n\n Daytime member helpline        1800-4032-222                  8 AM – 8 PM', '2.2 Night Helpline\n\n A dedicated night helpline operates from 8 PM to 8 AM on 1800-4032-777 for non-critical\n medical guidance. During daytime hours, members should use the standard helpline\n listed in Section 7.\n\n\n\n\n\nArogya Shield Residential Care Services — Member Policy Handbook v3.2                           Page 3', 'Critical patients are transported to the nearest partner hospital. Current partners are\n Sunrise Multispeciality Hospital (Gachibowli), Lotus Heart Institute (Madhapur), and\n Vishwas General Hospital (Kukatpally). Members may register one preferred hospital,\n which is honoured whenever the medical situation safely permits.\n\n 2.2 

## mutli query retrieval 


In [33]:
original_question = (
        'Can i use my unused physiotherapy'
        'Sessions during the next month?'
    )

print(original_question)


Can i use my unused physiotherapySessions during the next month?


In [34]:
queries = [
    'Can unused physiotherapy sessions be carried forward',
    'What happens to physiotherapy sessions that are not used within the billing month ?',
    'Do unused physiotherapy benefits expire at the end of the month ?',
    'Can remaining physiotherapy sessions be transferred to the next billing cycle ?',
]

print(queries)

['Can unused physiotherapy sessions be carried forward', 'What happens to physiotherapy sessions that are not used within the billing month ?', 'Do unused physiotherapy benefits expire at the end of the month ?', 'Can remaining physiotherapy sessions be transferred to the next billing cycle ?']


In [35]:
for index, query in enumerate(queries, start = 1):
    print(f'Query {index}: {query}')

Query 1: Can unused physiotherapy sessions be carried forward
Query 2: What happens to physiotherapy sessions that are not used within the billing month ?
Query 3: Do unused physiotherapy benefits expire at the end of the month ?
Query 4: Can remaining physiotherapy sessions be transferred to the next billing cycle ?


In [36]:
query_embedding = embedding_model.encode_query(
    queries,
    convert_to_numpy = True,
    normalize_embeddings = True,
)


print(query_embedding.shape)

(4, 384)


In [37]:
query_result = collection.query(
    query_embeddings=query_embedding,
    n_results=min(5, collection.count()),
    include = [
        'documents',
        'metadatas',
        'distances',
    ],
)

print(query_result)

{'ids': [['arogya_shield_policy_handbook-p 2-c05', 'arogya_shield_policy_handbook-p 2-c06', 'arogya_shield_policy_handbook-p 1-c02', 'arogya_shield_policy_handbook-p 3-c04', 'arogya_shield_policy_handbook-p 2-c03'], ['arogya_shield_policy_handbook-p 2-c05', 'arogya_shield_policy_handbook-p 2-c06', 'arogya_shield_policy_handbook-p 2-c03', 'arogya_shield_policy_handbook-p 6-c03', 'arogya_shield_policy_handbook-p 2-c02'], ['arogya_shield_policy_handbook-p 2-c05', 'arogya_shield_policy_handbook-p 1-c01', 'arogya_shield_policy_handbook-p 2-c06', 'arogya_shield_policy_handbook-p 2-c03', 'arogya_shield_policy_handbook-p 8-c05'], ['arogya_shield_policy_handbook-p 2-c05', 'arogya_shield_policy_handbook-p 2-c06', 'arogya_shield_policy_handbook-p 2-c03', 'arogya_shield_policy_handbook-p 4-c02', 'arogya_shield_policy_handbook-p 6-c03']], 'embeddings': None, 'documents': [['Physiotherapy sessions included with the Shield Premium plan must be used within the\n applicable billing month. Unused sessio

In [38]:
for query_index, query in enumerate(queries):
    print("\nQuery:", query)

    documents = query_result["documents"][query_index]
    metadatas = query_result["metadatas"][query_index]
    distances = query_result["distances"][query_index]

    for result_index, document in enumerate(documents):
        print("\nRank:", result_index + 1)
        print("Distance:", distances[result_index])
        print("Page:", metadatas[result_index]["page_number"])
        print("Document:", document)


Query: Can unused physiotherapy sessions be carried forward

Rank: 1
Distance: 0.40814948081970215
Page: 2
Document: Physiotherapy sessions included with the Shield Premium plan must be used within the
 applicable billing month. Unused sessions cannot be carried forward, transferred to
 another household, or converted into a cash benefit.

 1.1 Billing Cycle

Rank: 2
Distance: 0.7526671290397644
Page: 2
Document: 1.1 Billing Cycle

 Membership fees are billed on the 5th of every month. A grace period of 10 days applies,
 after which services are suspended until payment is received. Reactivation after
 suspension carries a fee of ₹250.





Arogya Shield Residential Care Services — Member Policy Handbook v3.2                           Page 2

Rank: 3
Distance: 0.7595170736312866
Page: 1
Document: Arogya Shield Residential Care Services provides round-the-clock emergency medical
 response, paramedic support, and preventive health services to residents of registered
 gated communities in

In [39]:
queries = ['If the ambulance arrives late, when does it count as a service failure', 
                     'and what refunds can the member recieve after a documented service failure ?']

In [40]:
query_embedding = embedding_model.encode_query(
    queries,
    convert_to_numpy = True,
    normalize_embeddings = True,
)

query_result = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3,
    include = [
        'documents',
        'metadatas',
        'distances',
    ],

)

In [41]:
for query_index, query in enumerate(queries):
    print("\nQuery:", query)

    documents = query_result["documents"][query_index]
    metadatas = query_result["metadatas"][query_index]
    distances = query_result["distances"][query_index]

    for result_index, document in enumerate(documents):
        print("\nRank:", result_index + 1)
        print("Distance:", distances[result_index])
        print("Page:", metadatas[result_index]["page_number"])
        print("Document:", document)


Query: If the ambulance arrives late, when does it count as a service failure

Rank: 1
Distance: 0.43568986654281616
Page: 3
Document: 2. Emergency Response Commitments

 Our response commitments below apply to all membership tiers within registered
 community premises.

    • First responder at the resident's door within 8 minutes of an emergency call.

    • Advanced life support ambulance on site within 15 minutes.

Rank: 2
Distance: 0.5011294484138489
Page: 3
Document: • Advanced life support ambulance on site within 15 minutes.

    • Handover to a partner hospital emergency department within 40 minutes for critical
    cases.

 Every registered tower is equipped with one automated external defibrillator (AED)
 placed in the ground-floor lobby, inspected on the first Monday of every month.

Rank: 3
Distance: 0.5915716886520386
Page: 8
Document: Complaints may be raised through the member portal, the mobile app, or the community
 service desk. Every complaint receives a ticket num

## Fuse and deduplicate multi-query results



In [42]:
print(queries)

['If the ambulance arrives late, when does it count as a service failure', 'and what refunds can the member recieve after a documented service failure ?']


In [ ]:
RRF_K = 60

fused_results: dict[str, dict[str, Any]] = {}

for query_index, query in enumerate(queries):

    documents = query_result['documents'][query_index]
    metadatas = query_result['metadatas'][query_index]
    distances = query_result['distances'][query_index]

    for rank, (document, metadata, distance) in enumerate(
        zip(documents, metadatas, distances),start = 1
    ):

        chunk_id = metadata['chunk_id']

        if chunk_id not in fused_results:
            fused_results[chunk_id] = {
                'document': document,
                'metadata': metadata,
                'rrf_score': 0.0,
                'best_distance': float(distance),
                'matched_queries': [],
            }
        fused_results[chunk_id]['rrf_score'] += 1 / (RRF_K + rank)

        fused_results[chunk_id]['best_distance'] = min(fused_results[chunk_id]['best_distance'], float(distance))

        fused_results[chunk_id]['matched_queries'].append(query)


print('Unique chunks after fusion', len(fused_results))
        




Unique chunks after fusion 6


In [44]:
from pprint import pprint
for chunk_id, result in fused_results.items():
    print("=" * 100)
    print("Chunk id         :", chunk_id)
    print("RRF score        :", f"{result['rrf_score']:.6f}")
    print("Best distance    :", f"{result['best_distance']:.4f}")
    print("Matched queries:")
    print(result['matched_queries'])
    print("\nDocument:")
    print(result['document'])
    print()

Chunk id         : arogya_shield_policy_handbook-p 3-c01
RRF score        : 0.016393
Best distance    : 0.4357
Matched queries:
['If the ambulance arrives late, when does it count as a service failure']

Document:
2. Emergency Response Commitments

 Our response commitments below apply to all membership tiers within registered
 community premises.

    • First responder at the resident's door within 8 minutes of an emergency call.

    • Advanced life support ambulance on site within 15 minutes.

Chunk id         : arogya_shield_policy_handbook-p 3-c02
RRF score        : 0.016129
Best distance    : 0.5011
Matched queries:
['If the ambulance arrives late, when does it count as a service failure']

Document:
• Advanced life support ambulance on site within 15 minutes.

    • Handover to a partner hospital emergency department within 40 minutes for critical
    cases.

 Every registered tower is equipped with one automated external defibrillator (AED)
 placed in the ground-floor lobby, in

In [45]:
fused_ranking = sorted(
    fused_results.values(),
    key=lambda result: (
        result["rrf_score"],
        -result["best_distance"],
    ),
    reverse=True,
)

print("Final fused ranking created.")
print("Total ranked chunks:", len(fused_ranking))

Final fused ranking created.
Total ranked chunks: 6


In [46]:
for final_rank, result in enumerate(fused_ranking, start=1):
    print("=" * 100)
    print("Final Rank    :", final_rank)
    print("Chunk ID      :", result["metadata"]["chunk_id"])
    print("Page Number   :", result["metadata"].get("page_number"))
    print("RRF Score     :", f"{result['rrf_score']:.6f}")
    print("Best Distance :", f"{result['best_distance']:.4f}")
    print("Matched Queries:")
    
    for matched_query in result["matched_queries"]:
        print(" -", matched_query)

    print("\nDocument:")
    print(result["document"])
    print()

Final Rank    : 1
Chunk ID      : arogya_shield_policy_handbook-p 1-c03
Page Number   : 1
RRF Score     : 0.016393
Best Distance : 0.3783
Matched Queries:
 - and what refunds can the member recieve after a documented service failure ?

Document:
emergency response commitments, refund and cancellation rules, and grievance
 procedures applicable to all members.

Final Rank    : 2
Chunk ID      : arogya_shield_policy_handbook-p 3-c01
Page Number   : 3
RRF Score     : 0.016393
Best Distance : 0.4357
Matched Queries:
 - If the ambulance arrives late, when does it count as a service failure

Document:
2. Emergency Response Commitments

 Our response commitments below apply to all membership tiers within registered
 community premises.

    • First responder at the resident's door within 8 minutes of an emergency call.

    • Advanced life support ambulance on site within 15 minutes.

Final Rank    : 3
Chunk ID      : arogya_shield_policy_handbook-p 6-c02
Page Number   : 6
RRF Score     : 0.0

## Now we are going to create the final evidence which we will pass to the local llm

In [47]:
FINAL_TOP_K = 5

top_fused_results = fused_ranking[:FINAL_TOP_K]

print("Selected top fused chunks:", len(top_fused_results))

Selected top fused chunks: 5


In [48]:
final_evidence = []

for result in top_fused_results:
    final_evidence.append(
        {
            'chunk_id': result['metadata']['chunk_id'],
            'page_number': result['metadata'].get("page_number"),
            'document': result['document'],
            'rrf_score': result['rrf_score'],
            'best_distance': result['best_distance'],
            'matched_queries': result['matched_queries'],
        }
    )


print('Final evidence prepeared:', len(final_evidence))

Final evidence prepeared: 5


In [49]:
for index, evidence in enumerate(final_evidence, start=1):
    print("=" * 100)
    print("Evidence Number :", index)
    print("Chunk ID        :", evidence["chunk_id"])
    print("Page Number     :", evidence["page_number"])
    print("RRF Score       :", f"{evidence['rrf_score']:.6f}")
    print("Best Distance   :", f"{evidence['best_distance']:.4f}")

    print("\nMatched Queries:")
    for matched_query in evidence["matched_queries"]:
        print(" -", matched_query)

    print("\nRetrieved Text:")
    print(evidence["document"])
    print()

Evidence Number : 1
Chunk ID        : arogya_shield_policy_handbook-p 1-c03
Page Number     : 1
RRF Score       : 0.016393
Best Distance   : 0.3783

Matched Queries:
 - and what refunds can the member recieve after a documented service failure ?

Retrieved Text:
emergency response commitments, refund and cancellation rules, and grievance
 procedures applicable to all members.

Evidence Number : 2
Chunk ID        : arogya_shield_policy_handbook-p 3-c01
Page Number     : 3
RRF Score       : 0.016393
Best Distance   : 0.4357

Matched Queries:
 - If the ambulance arrives late, when does it count as a service failure

Retrieved Text:
2. Emergency Response Commitments

 Our response commitments below apply to all membership tiers within registered
 community premises.

    • First responder at the resident's door within 8 minutes of an emergency call.

    • Advanced life support ambulance on site within 15 minutes.

Evidence Number : 3
Chunk ID        : arogya_shield_policy_handbook-p 6-c02

In [59]:
user_question = (
    "If an advanced life support ambulance arrives after 35 minutes, "
    "does this count as a documented service failure, "
    "and what refund can the member receive?"
)

queries = [
    user_question,
    "What is the committed arrival time for an advanced life support ambulance?",
    "What response time is considered a documented service failure?",
    "What happens when response time exceeds twice the committed limit?",
    "What refund is given for a documented service failure?",
]

for index, query in enumerate(queries, start=1):
    print(f"Query {index}: {query}")

Query 1: If an advanced life support ambulance arrives after 35 minutes, does this count as a documented service failure, and what refund can the member receive?
Query 2: What is the committed arrival time for an advanced life support ambulance?
Query 3: What response time is considered a documented service failure?
Query 4: What happens when response time exceeds twice the committed limit?
Query 5: What refund is given for a documented service failure?


In [60]:
query_embeddings = embedding_model.encode_query(
    queries,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

query_result = collection.query(
    query_embeddings=query_embeddings.tolist(),
    n_results=min(8, collection.count()),
    include=[
        "documents",
        "metadatas",
        "distances",
    ],
)

print("Queries searched:", len(queries))
print("Results per query:", len(query_result["documents"][0]))

Queries searched: 5
Results per query: 8


In [61]:
target_chunk_id = "arogya_shield_policy_handbook-p 5-c02"

target_chunk_found = False

for query_index, query in enumerate(queries):

    documents = query_result["documents"][query_index]
    metadatas = query_result["metadatas"][query_index]
    distances = query_result["distances"][query_index]

    for rank, (document, metadata, distance) in enumerate(
        zip(documents, metadatas, distances),
        start=1,
    ):
        if metadata["chunk_id"] == target_chunk_id:
            target_chunk_found = True

            print("\nTarget refund chunk found")
            print("Query    :", query)
            print("Rank     :", rank)
            print("Distance :", f"{float(distance):.4f}")
            print("Page     :", metadata.get("page_number"))
            print("\nText:")
            print(document)

print("\nTarget chunk retrieved:", target_chunk_found)


Target refund chunk found
Query    : If an advanced life support ambulance arrives after 35 minutes, does this count as a documented service failure, and what refund can the member receive?
Rank     : 8
Distance : 0.5874
Page     : 5

Text:
• Permanent relocation outside all registered communities, with proof of address
   change (pro-rata refund for unused full months).

    • Documented service failure, meaning a response time exceeding twice the
   committed limits in Section 2 (refund of that month's fee).

Target refund chunk found
Query    : What response time is considered a documented service failure?
Rank     : 3
Distance : 0.5832
Page     : 5

Text:
• Permanent relocation outside all registered communities, with proof of address
   change (pro-rata refund for unused full months).

    • Documented service failure, meaning a response time exceeding twice the
   committed limits in Section 2 (refund of that month's fee).

Target refund chunk found
Query    : What happens when 

In [62]:
target_chunk_id = "arogya_shield_policy_handbook-p 5-c02"

target_chunk_found = False

for query_index, query in enumerate(queries):

    documents = query_result["documents"][query_index]
    metadatas = query_result["metadatas"][query_index]
    distances = query_result["distances"][query_index]

    for rank, (document, metadata, distance) in enumerate(
        zip(documents, metadatas, distances),
        start=1,
    ):
        if metadata["chunk_id"] == target_chunk_id:
            target_chunk_found = True

            print("\nTarget refund chunk found")
            print("Query    :", query)
            print("Rank     :", rank)
            print("Distance :", f"{float(distance):.4f}")
            print("Page     :", metadata.get("page_number"))
            print("\nText:")
            print(document)

print("\nTarget chunk retrieved:", target_chunk_found)


Target refund chunk found
Query    : If an advanced life support ambulance arrives after 35 minutes, does this count as a documented service failure, and what refund can the member receive?
Rank     : 8
Distance : 0.5874
Page     : 5

Text:
• Permanent relocation outside all registered communities, with proof of address
   change (pro-rata refund for unused full months).

    • Documented service failure, meaning a response time exceeding twice the
   committed limits in Section 2 (refund of that month's fee).

Target refund chunk found
Query    : What response time is considered a documented service failure?
Rank     : 3
Distance : 0.5832
Page     : 5

Text:
• Permanent relocation outside all registered communities, with proof of address
   change (pro-rata refund for unused full months).

    • Documented service failure, meaning a response time exceeding twice the
   committed limits in Section 2 (refund of that month's fee).

Target refund chunk found
Query    : What happens when 

In [63]:
RRF_K = 60

fused_results: dict[str, dict[str, Any]] = {}

for query_index, query in enumerate(queries):

    documents = query_result["documents"][query_index]
    metadatas = query_result["metadatas"][query_index]
    distances = query_result["distances"][query_index]

    for rank, (document, metadata, distance) in enumerate(
        zip(documents, metadatas, distances),
        start=1,
    ):
        chunk_id = metadata["chunk_id"]
        distance = float(distance)

        if chunk_id not in fused_results:
            fused_results[chunk_id] = {
                "document": document,
                "metadata": metadata,
                "rrf_score": 0.0,
                "best_distance": distance,
                "matched_queries": [],
            }

        fused_results[chunk_id]["rrf_score"] += (
            1 / (RRF_K + rank)
        )

        fused_results[chunk_id]["best_distance"] = min(
            fused_results[chunk_id]["best_distance"],
            distance,
        )

        if query not in fused_results[chunk_id]["matched_queries"]:
            fused_results[chunk_id]["matched_queries"].append(query)

print("Unique chunks after fusion:", len(fused_results))

Unique chunks after fusion: 16


In [64]:
fused_ranking = sorted(
    fused_results.values(),
    key=lambda result: (
        -result["rrf_score"],
        result["best_distance"],
    ),
)

for final_rank, result in enumerate(fused_ranking, start=1):
    print(
        f"Rank {final_rank}"
        f" | Page {result['metadata'].get('page_number')}"
        f" | RRF {result['rrf_score']:.6f}"
        f" | Distance {result['best_distance']:.4f}"
    )
    print("Chunk ID:", result["metadata"]["chunk_id"])
    print()

Rank 1 | Page 8 | RRF 0.075837 | Distance 0.5468
Chunk ID: arogya_shield_policy_handbook-p 8-c02

Rank 2 | Page 3 | RRF 0.065053 | Distance 0.2906
Chunk ID: arogya_shield_policy_handbook-p 3-c01

Rank 3 | Page 5 | RRF 0.063366 | Distance 0.4342
Chunk ID: arogya_shield_policy_handbook-p 5-c02

Rank 4 | Page 1 | RRF 0.062756 | Distance 0.4896
Chunk ID: arogya_shield_policy_handbook-p 1-c03

Rank 5 | Page 6 | RRF 0.061824 | Distance 0.5122
Chunk ID: arogya_shield_policy_handbook-p 6-c02

Rank 6 | Page 6 | RRF 0.061786 | Distance 0.5117
Chunk ID: arogya_shield_policy_handbook-p 6-c01

Rank 7 | Page 3 | RRF 0.046964 | Distance 0.3175
Chunk ID: arogya_shield_policy_handbook-p 3-c02

Rank 8 | Page 7 | RRF 0.045928 | Distance 0.5580
Chunk ID: arogya_shield_policy_handbook-p 7-c01

Rank 9 | Page 6 | RRF 0.029631 | Distance 0.6454
Chunk ID: arogya_shield_policy_handbook-p 6-c03

Rank 10 | Page 5 | RRF 0.016129 | Distance 0.4634
Chunk ID: arogya_shield_policy_handbook-p 5-c03

Rank 11 | Page 4 | 

In [58]:
stored_data = collection.get(
    include=[
        "documents",
        "metadatas",
    ]
)

page_5_chunks = []

for document, metadata in zip(
    stored_data["documents"],
    stored_data["metadatas"],
):
    if metadata.get("page_number") == 5:
        page_5_chunks.append(
            {
                "document": document,
                "metadata": metadata,
            }
        )

print("Page 5 chunks stored in Chroma:", len(page_5_chunks))

for chunk in page_5_chunks:
    print("\n" + "=" * 100)
    print("Chunk ID:", chunk["metadata"]["chunk_id"])
    print(chunk["document"])

Page 5 chunks stored in Chroma: 4

Chunk ID: arogya_shield_policy_handbook-p 5-c01
4. Refund and Cancellation Policy

 4.1 Eligibility for Refunds

 Members may claim a refund of membership fees in the following situations:

    • Cancellation within the first 30 days of a new enrolment (full refund of monthly fees
    paid; the ₹999 enrolment fee is non-refundable).

Chunk ID: arogya_shield_policy_handbook-p 5-c02
• Permanent relocation outside all registered communities, with proof of address
   change (pro-rata refund for unused full months).

    • Documented service failure, meaning a response time exceeding twice the
   committed limits in Section 2 (refund of that month's fee).

Chunk ID: arogya_shield_policy_handbook-p 5-c03
Refund claims must be submitted through the member portal or in writing at the
 community service desk. Claims made verbally, including over the helpline, are not
 treated as formal refund requests.

 Note: the approval process and payment timelines for eli

In [65]:
FINAL_TOP_K = min(6, len(fused_ranking))

top_fused_results = fused_ranking[:FINAL_TOP_K]

final_evidence = []

for result in top_fused_results:
    final_evidence.append(
        {
            "chunk_id": result["metadata"]["chunk_id"],
            "page_number": result["metadata"].get("page_number"),
            "document": result["document"],
            "rrf_score": result["rrf_score"],
            "best_distance": result["best_distance"],
            "matched_queries": result["matched_queries"],
        }
    )

print("Final evidence prepared:", len(final_evidence))

Final evidence prepared: 6


In [66]:
combined_evidence_text = " ".join(
    evidence["document"].lower()
    for evidence in final_evidence
)

has_ambulance_rule = (
    "advanced life support ambulance" in combined_evidence_text
    and "15 minutes" in combined_evidence_text
)

has_service_failure_rule = (
    "documented service failure" in combined_evidence_text
    and "twice the" in combined_evidence_text
    and "committed" in combined_evidence_text
)

has_refund_rule = (
    "refund" in combined_evidence_text
    and "month's fee" in combined_evidence_text
)

print("Ambulance rule found      :", has_ambulance_rule)
print("Service-failure rule found:", has_service_failure_rule)
print("Refund rule found         :", has_refund_rule)

assert has_ambulance_rule, "Ambulance commitment was not retrieved."
assert has_service_failure_rule, "Service-failure rule was not retrieved."
assert has_refund_rule, "Refund rule was not retrieved."

print("All required evidence was retrieved.")

Ambulance rule found      : True
Service-failure rule found: True
Refund rule found         : True
All required evidence was retrieved.


## Now pull the local LLM from Ollama and pass the context to get the answer 


In [50]:
import ollama

In [67]:
test_response = ollama.chat(
    model = 'qwen3.5:4b',
    messages = [
        {
            'role': 'user',
            'content': 'Reply with exactly: Qwen is working locally',
        }
    ],
    think = False,
)

print(test_response['message']['content'])

Qwen is working locally


In [69]:
context_blocks = []

for evidence_number, evidence in enumerate(final_evidence, start = 1):
    context_block = (
        f"[Evidence {evidence_number}]\n"
        f"Page: {evidence['page_number']}\n"
        f"Chunk ID: {evidence['chunk_id']}\n"
        f"Text:\n{evidence['document']}"
    )

    context_blocks.append(context_block)

context_text = "\n\n".join(context_blocks)

print(context_text)

[Evidence 1]
Page: 8
Chunk ID: arogya_shield_policy_handbook-p 8-c02
Text:
Complaints may be raised through the member portal, the mobile app, or the community
 service desk. Every complaint receives a ticket number within 4 working hours. Standard
 complaints are resolved within 3 working days; complaints involving clinical care are
 reviewed by the Medical Advisory Board and resolved within 15 working days.

[Evidence 2]
Page: 3
Chunk ID: arogya_shield_policy_handbook-p 3-c01
Text:
2. Emergency Response Commitments

 Our response commitments below apply to all membership tiers within registered
 community premises.

    • First responder at the resident's door within 8 minutes of an emergency call.

    • Advanced life support ambulance on site within 15 minutes.

[Evidence 3]
Page: 5
Chunk ID: arogya_shield_policy_handbook-p 5-c02
Text:
• Permanent relocation outside all registered communities, with proof of address
   change (pro-rata refund for unused full months).

    • Document

In [70]:
system_prompt = """
You are a careful policy-document assistant.

Answer the question using only the retrieved evidence provided by the user.

Rules:
1. Do not use outside knowledge.
2. Do not invent information.
3. Cite factual claims using the page number, for example [Page 3].
4. If information from multiple evidence blocks is required, combine it carefully.
5. Show calculations clearly when necessary.
6. If the evidence is insufficient, say:
   "I could not find enough evidence in the provided document."
7. Keep the answer clear and direct.
"""

In [72]:
user_prompt = f"""
QUESTION:
{user_question}

RETRIEVED EVIDENCE:
{context_text}

INSTRUCTION:
Answer the question using only the retrieved evidence.
Cite the relevant page numbers in square brackets, such as [Page 3].
"""

In [73]:
response = ollama.chat(
    model = 'qwen3.5:4b',
    messages = [
        {
            'role': 'system',
            'content': system_prompt,
        },
        {
            'role': 'user',
            'content': user_prompt,
        }
    ],
    think = False,
    options = {
        'temperature': 0.1,
    },
)

In [74]:
print(response)

model='qwen3.5:4b' created_at='2026-07-27T08:41:58.783839Z' done=True done_reason='stop' total_duration=8791951200 load_duration=4747604300 prompt_eval_count=723 prompt_eval_duration=495078000 eval_count=189 eval_duration=3540511000 message=Message(role='assistant', content="Based on the provided evidence, here is the answer regarding your service failure and refund eligibility:\n\n**1. Does a 35-minute arrival count as a documented service failure?**\nYes. According to Section 2 of the policy, an advanced life support ambulance must be on site within **15 minutes** [Page 3]. A response time exceeding twice this committed limit (which equals $15 \\text{ minutes} \\times 2 = 30$ minutes) constitutes a documented service failure. Since 35 minutes exceeds the 30-minute threshold, it qualifies as a documented service failure [Page 5].\n\n**2. What refund can the member receive?**\nA member experiencing a documented service failure is entitled to **a refund of that month's fee**. This appli

In [75]:
print(response['message']['content'])

Based on the provided evidence, here is the answer regarding your service failure and refund eligibility:

**1. Does a 35-minute arrival count as a documented service failure?**
Yes. According to Section 2 of the policy, an advanced life support ambulance must be on site within **15 minutes** [Page 3]. A response time exceeding twice this committed limit (which equals $15 \text{ minutes} \times 2 = 30$ minutes) constitutes a documented service failure. Since 35 minutes exceeds the 30-minute threshold, it qualifies as a documented service failure [Page 5].

**2. What refund can the member receive?**
A member experiencing a documented service failure is entitled to **a refund of that month's fee**. This applies specifically when there is no permanent relocation outside registered communities (which would result in a pro-rata refund) [Page 5].
